In [1]:
#!/usr/bin/env python3
import os
import sys
import ROOT

ROOT.gROOT.SetBatch(True)       # no GUI
ROOT.gStyle.SetOptStat(0)

DEFAULT_FILE = "/w/hallb-scshelf2102/clas12/valerii/multiPi0/pass2_v3/statistics/pi0_nPions_ratio.root"

def is_hist(obj) -> bool:
    # Covers TH1* (and friends). Extend if you want TH2/TH3.
    return bool(obj.InheritsFrom("TH1"))

def draw_and_save(h, out_dir: str):
    # Basic styling
    h.SetLineWidth(2)
    h.SetMarkerStyle(20)
    h.SetMarkerSize(0.8)

    # Nice Y range for ratio-like histos
    if h.GetMaximum() > 0:
        h.SetMinimum(0)

    c = ROOT.TCanvas("c_" + h.GetName(), "", 900, 700)
    c.SetGrid()
    h.Draw("E1")   # draw with errors
    out_path = os.path.join(out_dir, f"{h.GetName()}.png")
    c.SaveAs(out_path)
    c.Close()

def visit_dir(tdir, out_dir: str):
    # Iterate keys; recurse into subdirectories
    keys = tdir.GetListOfKeys()
    if not keys:
        return
    for key in keys:
        obj = key.ReadObj()
        if obj.InheritsFrom("TDirectory"):
            sub_out = os.path.join(out_dir, obj.GetName())
            os.makedirs(sub_out, exist_ok=True)
            visit_dir(obj, sub_out)
        elif is_hist(obj):
            draw_and_save(obj, out_dir)

def main():
    in_file = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_FILE
    if not os.path.exists(in_file):
        sys.exit(f"[ERROR] File not found: {in_file}")

    out_dir = os.path.join(os.path.dirname(in_file), "plots_ratio")
    os.makedirs(out_dir, exist_ok=True)

    f = ROOT.TFile.Open(in_file, "READ")
    if not f or f.IsZombie():
        sys.exit(f"[ERROR] Failed to open ROOT file: {in_file}")

    visit_dir(f, out_dir)
    f.Close()
    print(f"[OK] Saved histograms to: {out_dir}")

if __name__ == "__main__":
    main()


SystemExit: [ERROR] File not found: -f

/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
